In [1]:
%pip install python-dotenv pandas firecrawl-py

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv
import pandas
from firecrawl import Firecrawl
from firecrawl.v2.types import ScrapeOptions
from pydantic import BaseModel, Field
from typing import Optional, Literal
import json

In [3]:
# Load variables from the .env file into the environment
exists=load_dotenv(dotenv_path=".pricecompare_config", override=True)

if not exists:
    with open(".pricecompare_config",'w') as config:
        config.write("SPREADSHEET_PATH=\"spreadsheet.json\"")

api_key = os.getenv("FIRECRAWL_API_KEY")
location = os.getenv("LOCATION")
retailers = os.getenv("STORES","").split(',')
items = os.getenv("GROCERIES","").split(',')
destination = os.getenv("SPREADSHEET_PATH")

print(retailers)
print(items)

# Use your api_key normally below
print(f"Loaded key successfully: {api_key[:4]}...") 

firecrawl = Firecrawl()

class Product(BaseModel):
    item: str = Field(description="Display name of requested item")
    price: Optional[float] = Field(None, description="Total price in USD.")
    quantity: Optional[float] = Field(None, description="Numeric amount only, e.g. 42 for '42 oz'. For multipacks, the total amount if stated, otherwise the amount per item.")
    unit: Optional[Literal["oz", "lb", "g", "kg", "fl oz", "ml", "l", "ct"]] = Field(None, description="Unit exactly as printed, using the closest option")


['HEB', 'H-Mart']
['Oatmeal', 'Bananas', 'Oranges']
Loaded key successfully: fc-2...


In [ ]:
class R:
    def __init__(self):
        self.web=[{'item': 'H-E-B Instant Oatmeal Variety Pack', 'price': 3.35, 'quantity': 18, 'unit': 'ct'}]

class priceCompare:
    def __init__(self,retailers=retailers,items=items,path=destination):
        try:
            with open(path) as f:
                df = pandas.read_json(f)
                df = df.reindex(
                    index=df.index.union(retailers),
                    columns=df.columns.union(items),
                    fill_value=17
                )
        except:
            df = pandas.DataFrame(data=None,index=retailers,columns=items)

        self.data=df.astype('object')
        self.path=path

    def save(self):
        self.data.to_json(path_or_buf=self.path,mode='w')

    def print(self):
        display(self.data)

    def update_single(self:priceCompare, retailer:str,item:str,limit=1):
        if limit < 1:
            limit = 1    
        TO_OZ = {"oz": 1, "lb": 16, "g": 0.03527, "kg": 35.27}

        #result = R()

        result = firecrawl.search(f"{item} lowest price per unit at {retailer} near {location}",limit=limit,
                                scrape_options=ScrapeOptions(
                                    formats=[{
                                        "type":"json",
                                        "schema": Product.model_json_schema(),
                                        "prompt": "Extract the product details from this page."
                                    }]
                                ))
        results_table = []

        for r in result.web:
            js = r.json
            print(js)
            if js["quantity"] and js["unit"]:
                if js["unit"]!="ct":
                    js["quantity"] = js["quantity"] * TO_OZ[js["unit"]]
                    js["unit"] = "oz"
            else:
                js["quantity"] = 1
                js["unit"] = 'ct'

            results_table.append(js)

        results_table=pandas.DataFrame(results_table)
        rows_in_oz = results_table[results_table["unit"]=="oz"]
        rows_in_ct = results_table[results_table["unit"]=="ct"]
        if len(rows_in_oz) > 0:
            #If weight available, use weight.
            results_table=rows_in_oz
        elif len(rows_in_ct) > 0:
            #If no weights available, and units available, use units.
            results_table=rows_in_ct
        else:
            #If no units available, assume sold as unit of one count
            results_table["quantity"] = 1
            results_table["unit"] = "ct"
        
        results_table["unit_price"]=results_table["price"]/results_table["quantity"]

        if limit == 1:
            output=results_table.iloc[0].to_dict()
        else:
            try:
                result_min=results_table.loc[results_table["unit_price"].idxmin(skipna=True)]
                output=result_min.to_dict()
            except:
                pass
            output=None

        #output = f"{retailer},{item}"

        print(f"{retailer}, {item}, updating with {output}")    
        self.data.at[retailer,item]=output
        print(f"{retailer}, {item}, updated")

    def update_row(self,retailer,items:list[str]):
        pandas.Series(items).apply(func=(lambda item: self.update_single(retailer,item)))


    def update_multi(self,retailers=retailers,items=items):
        pandas.Series(retailers).apply(func=(lambda retailer: self.update_row(retailer,items)))



pc=priceCompare()

,Oatmeal,Bananas,Oranges
HEB,"{'item': 'H-E-B Instant Oatmeal Variety Pack',...","{'item': 'H-E-B Instant Oatmeal Variety Pack',...","{'item': 'H-E-B Instant Oatmeal Variety Pack',..."
H-Mart,"{'item': 'H-E-B Instant Oatmeal Variety Pack',...","{'item': 'H-E-B Instant Oatmeal Variety Pack',...","{'item': 'H-E-B Instant Oatmeal Variety Pack',..."


{'item': 'H‑E‑B Instant Oatmeal Variety Pack', 'price': 3.35, 'quantity': 18, 'unit': 'ct'}
HEB, Oatmeal, updating with {'item': 'H‑E‑B Instant Oatmeal Variety Pack', 'price': 3.35, 'quantity': 18, 'unit': 'ct', 'unit_price': 0.18611111111111112}
HEB, Oatmeal, updated
{'item': 'H-E-B Frozen Banana Slices', 'price': 3.68, 'quantity': 24, 'unit': 'oz'}
HEB, Bananas, updating with {'item': 'H-E-B Frozen Banana Slices', 'price': 3.68, 'quantity': 24, 'unit': 'oz', 'unit_price': 0.15333333333333335}
HEB, Bananas, updated
{'item': 'H‑E‑B Texas Roots Fresh Oranges', 'price': 8.98, 'quantity': 10, 'unit': 'lb'}
HEB, Oranges, updating with {'item': 'H‑E‑B Texas Roots Fresh Oranges', 'price': 8.98, 'quantity': 160, 'unit': 'oz', 'unit_price': 0.056125}
HEB, Oranges, updated
{'item': 'Hetbahn Cooked Sprouted Brown Rice Box', 'price': 24.99, 'quantity': 12, 'unit': 'ct'}
H-Mart, Oatmeal, updating with {'item': 'Hetbahn Cooked Sprouted Brown Rice Box', 'price': 24.99, 'quantity': 12, 'unit': 'ct', 

,Oatmeal,Bananas,Oranges
HEB,"{'item': 'H‑E‑B Instant Oatmeal Variety Pack',...","{'item': 'H-E-B Frozen Banana Slices', 'price'...","{'item': 'H‑E‑B Texas Roots Fresh Oranges', 'p..."
H-Mart,{'item': 'Hetbahn Cooked Sprouted Brown Rice B...,{'item': 'Hetbahn Cooked Sprouted Brown Rice B...,{'item': 'Hetbahn Cooked Sprouted Brown Rice B...
